In [43]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_3120/3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [48]:
from google.colab import files

uploaded = files.upload()

Saving constitution.pdf to constitution (1).pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_file(data):
    loader = PyPDFLoader(data)
    documents = loader.load()
    return documents

In [ ]:
import os
print(os.path.exists("constitution.pdf"))

True


In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf

In [ ]:
extracted_data = load_pdf_file("constitution.pdf")

In [ ]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 2047


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [ ]:
embeddings=download_hugging_face_embeddings()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
query_result=embeddings.embed_query("This is my first project")
print("Length",len(query_result))

Length 384


In [51]:
import os

# This one code works for BOTH Colab and Render / GitHub
try:
    # For Colab
    from google.colab import userdata
    groq_api_key = userdata.get('GROQ_API_KEY')
except:
    # For Laptop / Render / HuggingFace
    groq_api_key = os.getenv("GROQ_API_KEY")

print("Key loaded:", "Yes" if groq_api_key else "No - Add it in Secrets!")

Key loaded: Yes


In [ ]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "constition-chatbot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully.")
else:
    print("Index already exists. Using existing index.")

Index already exists. Using existing index.


In [ ]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is right to freedom")

In [ ]:
retrieved_docs

[Document(id='49cf5307-11f9-4ef0-9814-7c6de12f7920', metadata={'creationdate': '', 'creator': 'PyPDF', 'moddate': '2024-07-01T11:20:33+00:00', 'page': 39.0, 'page_label': '40', 'producer': 'iLovePDF', 'source': 'constitution.pdf', 'total_pages': 402.0}, page_content='(4) No person holding any office of profit or trust under the State shall, \nwithout the consent of the President, accept any present, emolument, or office \nof any kind from or under any foreign State.\nRight to Freedom\n19. Protection of certain rights regarding freedom of speech, etc. —\n(1) All citizens shall have the right—\n(a) to freedom of speech and expression;\n(b) to assemble peaceably and without arms;\n(c) to form associations or unions2[or co-operative societies];'),
 Document(id='38659dad-59b7-4cc9-bb3b-590abc019c34', metadata={'creationdate': '', 'creator': 'PyPDF', 'moddate': '2024-07-01T11:20:33+00:00', 'page': 39.0, 'page_label': '40', 'producer': 'iLovePDF', 'source': 'constitution.pdf', 'total_pages': 

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0
)

In [ ]:
# ChatPromptTemplate: Creates a structured prompt template for sending
# system instructions and user messages to the LLM.
from langchain_core.prompts import ChatPromptTemplate
# RunnablePassthrough: Passes the input data through a LangChain chain
# without changing it.
from langchain_core.runnables import RunnablePassthrough
# StrOutputParser: Converts the LLM's response into a simple string.
from langchain_core.output_parsers import StrOutputParser


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "Context:\n{context}"
)

# Creates a chat prompt containing a system message and a human/user message.
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("If the Government imposes greater control and censorship over social media platforms to curb fake news, which fundamental rights under Article 19(1)(a), Article 19(2), and Article 21 (Right to Privacy from Puttaswamy judgment) will come into conflict, and how did the Supreme Court in Shreya Singhal v. Union of India resolve this balance?")
print(response)

Greater control and censorship would clash the **freedom of speech and expression** under Article 19(1)(a) with the **reasonable‑restrictions** clause of Article 19(2), and it could also impinge the **right to privacy** protected under Article 21 (as held in the Puttaswamy judgment). In *Shreya Singhal v. Union of India*, the Supreme Court struck down Section 66A of the IT Act as vague and overbroad, holding that any restriction on expression must be narrowly tailored, clear, and proportionate to a legitimate aim. The Court therefore upheld that while the state may impose restrictions under Article 19(2), they must not violate the fundamental rights to free expression and privacy.


In [ ]:
response=rag_chain.invoke("explain any 1 fundamental right")
print(response)





**Article 14 – Equality before the law**  
Article 14 guarantees that every person is treated equally by the law and that no one is denied equal protection of the law. It prohibits arbitrary discrimination by the State and ensures that laws apply uniformly to all citizens. This right forms the basis for equal treatment in legal proceedings and public administration.


In [ ]:
!pip install -q langchain langchain-huggingface langchain-pinecone langchain-groq gradio pinecone

In [46]:
import gradio as gr
import os

# Your rag_chain must be created in SAME cell or above
# Example: make sure this part is there before chatbot function
# If you already created rag_chain in previous cell, copy that code here

def samvidhan_chatbot(question):
    if not question or not question.strip():
        return "Please ask a question"
    try:
        # rag_chain must exist
        answer = rag_chain.invoke(question)
        return answer
    except Exception as e:
        return f"Error: {e}"

demo = gr.Interface(
    fn=samvidhan_chatbot,
    inputs=gr.Textbox(label="Ask about Constitution", placeholder="What is Article 21?"),
    outputs=gr.Textbox(label="Answer", lines=8),
    title="Samvidhan AI",
    description="Indian Constitution Assistant"
)

demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True  # <-- This gives you the public link
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3954d94beb5b178deb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
%%writefile app.py
import os
import gradio as gr
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq

# Read keys from HuggingFace Secrets (safe, no error)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Loading model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = PineconeVectorStore(index_name="samvidhan-index", embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

def chatbot(question):
    if not question.strip():
        return "Please ask a question"
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    prompt = f"You are Samvidhan AI. Answer from context only.\nContext: {context}\nQuestion: {question}\nAnswer:"
    return llm.invoke(prompt).content

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(label="Ask about Constitution", placeholder="What is Article 21?"),
    outputs=gr.Textbox(label="Answer", lines=8),
    title="Samvidhan AI - Indian Constitution Assistant",
    description="Permanent version for GitHub & HuggingFace"
)

if __name__ == "__main__":
    demo.launch()

Writing app.py


In [ ]:
%%writefile requirements.txt
gradio
langchain
langchain-huggingface
langchain-pinecone
langchain-groq
pinecone
sentence-transformers
groq

Writing requirements.txt


In [ ]:
!git init
!git config --global user.email "you@example.com"
!git config --global user.name "SamvidhanAI"
!git add app.py requirements.txt
!git commit -m "Samvidhan AI - final working version"
# Create empty repo on github.com named Samvidhan-AI and copy its link
!git branch -M main
!git remote add origin https://github.com/YOUR_USERNAME/Samvidhan-AI.git
!git push -u origin main

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
[master (root-commit) ae468eb] Samvidhan AI - final working version
 2 files changed, 42 insertions(+)
 create mode 100644 app.py
 create mode 100644 requirements.txt
fatal: could not read Username for 'https://github.com': No such device or address


In [47]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/Samvidhan_AI"
!cp /content/app.py "/content/drive/MyDrive/Samvidhan_AI/"
!cp /content/requirements.txt "/content/drive/MyDrive/Samvidhan_AI/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
